In [ ]:
pip install pennylane

In [ ]:
pip install jax~=0.6.0 jaxlib~=0.6.0

In [ ]:
# ============================================================
# Warning Suppression for cleaner log output
# ============================================================
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# TensorFlow internal warnings
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

# PennyLane warnings
warnings.filterwarnings("ignore", module="pennylane")


In [ ]:
# ============================================================
# QUANTUM ABLATION STUDY
# Qubits × Circuit Depth × Encoding Strategy
# 
# ============================================================

import time
import pickle
import pennylane as qml
import tensorflow as tf
import numpy as np
import pandas as pd

from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM,
    Dense, Dropout, GlobalAveragePooling1D, Concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# ============================================================
# Reproducibility
# ============================================================
SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

# ============================================================
# GLOBAL HYPERPARAMETERS (SELF-CONTAINED)
# ============================================================
MAX_LEN     = 50
VOCAB_SIZE  = 10_000
BATCH_SIZE  = 128
EPOCHS      = 8
LR          = 1e-4

# ============================================================
# Ablation Grid
# ============================================================
QUBITS_LIST = [2, 4, 6]
DEPTH_LIST  = [1, 2]
ENCODINGS   = ["angle", "reupload"]

# ============================================================
# Load Tokenizer (from classical baseline)
# ============================================================
TOKENIZER_PATH = "/kaggle/input/classicaltokenizer/tokenizer.pkl"

with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

def encode(texts):
    return pad_sequences(
        tokenizer.texts_to_sequences(texts),
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

# ============================================================
# Load Sentiment140 (100k subset)
# ============================================================
def load_sentiment140(path, max_samples=100_000):
    cols = ["target", "id", "date", "query", "user", "text"]
    df = pd.read_csv(path, encoding="latin-1", header=None, names=cols)
    df = df[["target", "text"]]
    df["target"] = df["target"].replace({4: 1})

    df = (
        df.groupby("target", group_keys=False)
          .apply(lambda x: x.sample(max_samples // 2, random_state=SEED))
          .sample(frac=1.0, random_state=SEED)
          .reset_index(drop=True)
    )
    return df

FILE_PATH = "/kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv"
df = load_sentiment140(FILE_PATH)

X = encode(df.text.values)
y = df.target.values

# ============================================================
# Train / Val / Test Split (60 / 20 / 20)
# ============================================================
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED
)

print(f"[INFO] Dataset sizes → Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# ============================================================
# Quantum Ablation Layer
# ============================================================
class QuantumAblationLayer(tf.keras.layers.Layer):
    def __init__(self, n_qubits, depth, encoding):
        super().__init__()
        self.n_qubits = n_qubits
        self.depth = depth
        self.encoding = encoding
        self.dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(self.dev, interface="tf")
        def circuit(x):
            for d in range(depth):
                for i in range(n_qubits):
                    qml.RY(x[i], wires=i)
                    if encoding == "reupload":
                        qml.RY(x[i], wires=i)
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.circuit = circuit

    def call(self, x):
        x = tf.math.l2_normalize(x, axis=-1)

        def run(sample):
            q = self.circuit(sample)
            return tf.cast(tf.math.real(q), tf.float32)

        return tf.map_fn(
            run,
            x,
            fn_output_signature=tf.TensorSpec((self.n_qubits,), tf.float32)
        )

# ============================================================
# Model Builder
# ============================================================
def build_model(n_qubits, depth, encoding):
    inp = Input(shape=(MAX_LEN,))
    x = Embedding(VOCAB_SIZE, 128)(inp)
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = Dropout(0.5)(x)
    x = GlobalAveragePooling1D()(x)

    q = Dense(n_qubits, activation="tanh")(x)
    q = QuantumAblationLayer(n_qubits, depth, encoding)(q)

    fused = Concatenate()([x, q])
    fused = Dense(64, activation="relu")(fused)
    out = Dense(2, activation="softmax")(fused)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ============================================================
# Run Ablation
# ============================================================
results = []

for Q in QUBITS_LIST:
    for D in DEPTH_LIST:
        for ENC in ENCODINGS:
            print(f"\n▶ Running: Q={Q}, Depth={D}, Encoding={ENC}")

            model = build_model(Q, D, ENC)

            start = time.time()
            model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
                verbose=0
            )
            train_time = time.time() - start

            y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
            f1 = f1_score(y_test, y_pred)

            results.append({
                "qubits": Q,
                "depth": D,
                "encoding": ENC,
                "f1": f1,
                "train_time_sec": train_time
            })

# ============================================================
# Results
# ============================================================
df_ablation = pd.DataFrame(results)
print("\n=== QUANTUM ABLATION RESULTS ===")
print(df_ablation)

df_ablation.to_csv("quantum_ablation_results.csv", index=False)
